# Modelado unificado de las representaciones funcionales EEG

Este cuaderno realiza el modelado para la clasificación experimental HC vs PD_OFF a partir
de las representaciones generadas en los cuadernos `02A` y `02B`.

Se evalúan:
- **Enfoque A1:** una representación funcional por sujeto en el dominio de frecuencia (4-13 Hz).
- **Enfoque A2:** representaciones funcionales por época, con partición y agregación final por sujeto.
- **Enfoque B:** trayectorias funcionales theta y alfa sobre el orden normalizado de las épocas.
- **Línea base sin ADF:** características espectrales escalares por sujeto.

Cada representación funcional se evalúa en tres formas: coeficientes B-spline (`X_raw`),
coeficientes transformados por Cholesky (`X_chol`) y scores FPCA obtenidos a partir de
`X_chol` (`X_chol_fpca`).

La evaluación utiliza validación cruzada estratificada y agrupada por sujeto
(`StratifiedGroupKFold`), con 5 pliegues y las semillas 42, 123 y 2024. Las transformaciones
dependientes de los datos (FPCA, estandarización) se ajustan únicamente con las observaciones
de entrenamiento de cada partición.

Las métricas principales se reportan a nivel de sujeto: exactitud balanceada, AUC,
sensibilidad, especificidad y F1-score, como media ± desviación estándar sobre las 15
evaluaciones (5 pliegues × 3 semillas).

In [ ]:
# !pip -q install scikit-learn torch  # instalar con requirements.txt

## 1. Entorno e importaciones


In [ ]:
import os, pickle, warnings, time, itertools, json
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing  import StandardScaler
from sklearn.decomposition  import PCA
from sklearn.linear_model   import LogisticRegression
from sklearn.ensemble       import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score, f1_score, confusion_matrix,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## 2. Carga de las representaciones funcionales

Se cargan los datasets generados por los cuadernos `02A_UNM_enfoque_A.ipynb`
(A1 por sujeto y A2 por época) y `02B_UNM_enfoque_B.ipynb` (trayectorias theta-alfa).

In [ ]:
from pathlib import Path

import numpy as np
# Rutas portátiles: Google Drive solo se monta si USE_GOOGLE_DRIVE=1 (Colab).
import os
from pathlib import Path

if os.environ.get('USE_GOOGLE_DRIVE') == '1':
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
else:
    print('Ejecución local: se usan PARKINSON_EEG_ROOT y PARKINSON_EEG_DATA.')


_REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINAL_ROOT = Path(os.environ.get('PARKINSON_EEG_ROOT', _REPO_ROOT / 'artifacts')).expanduser().resolve()

A_SUBJ_PATH = FINAL_ROOT / '02A_enfoque_A' / 'enfoque_A_sujeto_dataset.npz'
A_EPOCH_PATH = FINAL_ROOT / '02A_enfoque_A' / 'enfoque_A_dataset.npz'
B_PATH = FINAL_ROOT / '02B_enfoque_B' / 'enfoque_B_dataset_theta_alpha.npz'

for path in (A_SUBJ_PATH, A_EPOCH_PATH, B_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'No se encuentra el dataset: {path}')

OUTPUT_DIR = FINAL_ROOT / '03_modelado'
RESULTS_DIR = OUTPUT_DIR / 'resultados'
FIG_DIR = OUTPUT_DIR / 'figuras'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

MAIN_EXPERIMENT = 'CTL_vs_PD_OFF'
all_datasets = {MAIN_EXPERIMENT: {}}

# Enfoque A1: una representación por sujeto.
_dAs = np.load(A_SUBJ_PATH, allow_pickle=False)
all_datasets[MAIN_EXPERIMENT]['A'] = {
    'X_raw': _dAs['X_raw'],
    'X_chol': _dAs['X_chol'],
    'y': _dAs['y'],
    'groups': _dAs['groups'],
}

# Enfoque A2: una representación por época.
_dAe = np.load(A_EPOCH_PATH, allow_pickle=False)
all_datasets[MAIN_EXPERIMENT]['A_epoca'] = {
    'X_raw': _dAe['X_raw'],
    'X_chol': _dAe['X_chol'],
    'y': _dAe['y'],
    'groups': _dAe['groups'],
}

# Enfoque B: trayectorias theta-alfa por sujeto.
_dB = np.load(B_PATH, allow_pickle=False)
all_datasets[MAIN_EXPERIMENT]['B'] = {
    'X_raw': _dB['X_raw'],
    'X_chol': _dB['X_chol'],
    'y': _dB['y'],
    'groups': _dB['groups'],
    'n_basis_per_band': [int(v) for v in _dB['n_basis_per_band']],
    'bands': [str(v) for v in _dB['bands']],
}

print('Carpeta de salida:', OUTPUT_DIR)
print('\nDatasets cargados:')
for enfoque, dataset in all_datasets[MAIN_EXPERIMENT].items():
    print(
        f'  {enfoque:8s}: '
        f'X_raw={dataset["X_raw"].shape} | '
        f'X_chol={dataset["X_chol"].shape} | '
        f'sujetos={len(np.unique(dataset["groups"]))} | '
        f'filas={len(dataset["y"])}'
    )


## 3. Configuración experimental


In [ ]:
REPRESENTATIONS = ['X_raw', 'X_chol', 'X_chol_fpca']

N_FOLDS = 5
SEEDS = [42, 123, 2024]

BATCH_SIZE = 32
EPOCHS = 80
PATIENCE = 15
WEIGHT_DECAY = 1e-4
DEFAULT_HP = {'lr': 1e-3, 'dropout': 0.3}

FPCA_VARIANCE_THRESHOLD = 0.95

print('Representaciones:', REPRESENTATIONS)
print(f'Umbral FPCA: {FPCA_VARIANCE_THRESHOLD*100:.0f}% de varianza explicada')


## 4. FPCA dentro de la validación

La FPCA se ajusta dentro de cada partición utilizando exclusivamente los datos de entrenamiento. En el Enfoque A se aplica por canal y en el Enfoque B de manera independiente por combinación canal-banda.


In [ ]:
def fpca_per_channel(
    X_train: np.ndarray,
    X_test: np.ndarray,
    variance_threshold: float = FPCA_VARIANCE_THRESHOLD,
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    """FPCA aplicada por canal independiente (Enfoque A).

    X_train, X_test: shape (n_obs, n_channels, n_basis_chol)

    Returns
    -------
    X_tr_fpca, X_te_fpca: shape (n_obs, n_channels, n_components)
    info: dict con n_components por canal y varianza explicada
    """
    n_tr, n_ch, n_basis = X_train.shape
    n_te = X_test.shape[0]

    # Primer paso: encontrar n_components por canal sobre el training set
    n_components_per_ch = []
    explained_var_per_ch = []
    pcas = []
    for ch in range(n_ch):
        pca_tmp = PCA()   # todos los componentes
        pca_tmp.fit(X_train[:, ch, :])
        cum_var = np.cumsum(pca_tmp.explained_variance_ratio_)
        n_comp = int(np.searchsorted(cum_var, variance_threshold) + 1)
        n_comp = min(max(n_comp, 1), n_basis)
        # Re-fit con n_comp componentes
        pca = PCA(n_components=n_comp)
        pca.fit(X_train[:, ch, :])
        pcas.append(pca)
        n_components_per_ch.append(n_comp)
        explained_var_per_ch.append(float(np.sum(pca.explained_variance_ratio_)))

    # Tomamos el MAX para que todas las matrices tengan la misma shape final
    # (los componentes adicionales se rellenan con 0 si un canal usa menos)
    n_comp_max = max(n_components_per_ch)

    X_tr_fpca = np.zeros((n_tr, n_ch, n_comp_max))
    X_te_fpca = np.zeros((n_te, n_ch, n_comp_max))
    for ch, pca in enumerate(pcas):
        n_c = n_components_per_ch[ch]
        X_tr_fpca[:, ch, :n_c] = pca.transform(X_train[:, ch, :])
        X_te_fpca[:, ch, :n_c] = pca.transform(X_test[:, ch, :])

    info = {
        'n_components_per_channel': n_components_per_ch,
        'explained_var_per_channel': explained_var_per_ch,
        'n_comp_max': n_comp_max,
    }
    return X_tr_fpca, X_te_fpca, info


def fpca_per_band_per_channel(
    X_train: np.ndarray,
    X_test: np.ndarray,
    n_basis_per_band: List[int],
    variance_threshold: float = FPCA_VARIANCE_THRESHOLD,
) -> Tuple[np.ndarray, np.ndarray, Dict]:
    """FPCA aplicada por banda y por canal (Enfoque B).

    X tiene shape (n_obs, n_channels, sum(n_basis_per_band)) — bandas concatenadas
    n_basis_per_band: lista de n_basis para cada banda (ej. [5,5,5,5,5])

    Aplica FPCA por separado sobre cada bloque de banda en cada canal.
    """
    n_tr, n_ch, _ = X_train.shape
    n_te = X_test.shape[0]
    n_bands = len(n_basis_per_band)

    # Indices de inicio/fin de cada banda en el eje de features
    band_starts = np.cumsum([0] + list(n_basis_per_band))

    # Resultado por banda separado, luego concatenamos
    X_tr_fpca_blocks = []
    X_te_fpca_blocks = []
    n_comp_total = 0
    info_per_band = {}

    for b_idx in range(n_bands):
        b_start = band_starts[b_idx]
        b_end   = band_starts[b_idx + 1]
        # Extraer bloque de la banda
        X_tr_band = X_train[:, :, b_start:b_end]   # (n_tr, n_ch, n_basis_band)
        X_te_band = X_test[:, :, b_start:b_end]

        # FPCA por canal dentro de esta banda
        X_tr_band_fpca, X_te_band_fpca, info_band = fpca_per_channel(
            X_tr_band, X_te_band, variance_threshold,
        )
        X_tr_fpca_blocks.append(X_tr_band_fpca)
        X_te_fpca_blocks.append(X_te_band_fpca)
        n_comp_total += info_band['n_comp_max']
        info_per_band[f'band_{b_idx}'] = info_band

    # Concatenar bandas en el eje de features
    X_tr_fpca = np.concatenate(X_tr_fpca_blocks, axis=2)
    X_te_fpca = np.concatenate(X_te_fpca_blocks, axis=2)

    info = {
        'per_band': info_per_band,
        'n_comp_total': n_comp_total,
    }
    return X_tr_fpca, X_te_fpca, info

## 5. Utilidades de validación, métricas y agregación por sujeto


In [ ]:
def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def standardize_per_fold(X_tr, X_te):
    sh_tr, sh_te = X_tr.shape, X_te.shape
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr.reshape(sh_tr[0], -1)).reshape(sh_tr)
    X_te_s = sc.transform(X_te.reshape(sh_te[0], -1)).reshape(sh_te)
    return X_tr_s, X_te_s


def aggregate_by_subject(y_proba, y_true, groups):
    unique = np.unique(groups)
    y_t, y_p = [], []
    for s in unique:
        m = groups == s
        y_t.append(int(y_true[m][0]))
        y_p.append(float(y_proba[m].mean()))
    return np.asarray(y_t), np.asarray(y_p)


def compute_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    try: auc = roc_auc_score(y_true, y_proba)
    except: auc = np.nan
    return {
        'balanced_acc': balanced_accuracy_score(y_true, y_pred),
        'auc': auc, 'sensitivity': sens, 'specificity': spec,
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }


def get_representation_data(experiment, approach, repr_key):
    """Devuelve (X, y, groups) y metadata para una representacion.

    Para X_chol_fpca devuelve X = X_chol (FPCA se aplica DENTRO del fold).
    """
    ds = all_datasets[experiment][approach]
    if repr_key in ['X_raw', 'X_chol']:
        X = ds[repr_key]
        needs_fpca = False
    elif repr_key == 'X_chol_fpca':
        X = ds['X_chol']   # punto de partida; FPCA se hace en el fold
        needs_fpca = True
    else:
        raise ValueError(f'Repr desconocida: {repr_key}')
    y = ds['y']
    groups = ds['groups']

    # Metadata extra para Enfoque B (n_basis_per_band para FPCA por banda)
    extra = {}
    if approach == 'B' and 'n_basis_per_band' in ds:
        extra['n_basis_per_band'] = list(ds['n_basis_per_band'])
        extra['bands'] = list(ds.get('bands', []))
    return X, y, groups, needs_fpca, extra

## 6. Diagnóstico FPCA del Enfoque A

Se resume el número de componentes retenidos por canal en las 15 particiones externas. Este bloque no entrena clasificadores; documenta la reducción dimensional utilizada posteriormente.


In [ ]:
# Diagnóstico FPCA del Enfoque A sobre las 15 particiones externas.
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedGroupKFold

ENFOQUE_DIAG = 'A'          # 'A' = Enfoque A por sujeto (el de la contribucion)
CH_NAMES     = ['C3', 'Cz', 'C4', 'P7', 'P8', 'P3', 'Pz', 'P4', 'O1', 'O2']
GUARDAR      = True

X_d, y_d, groups_d, _, _ = get_representation_data(MAIN_EXPERIMENT, ENFOQUE_DIAG, 'X_chol')
n_ch_d, n_basis_d = X_d.shape[1], X_d.shape[2]
ch_names = CH_NAMES if len(CH_NAMES) == n_ch_d else [f'ch{i}' for i in range(n_ch_d)]
print(f'Enfoque {ENFOQUE_DIAG}: X={X_d.shape} | sujetos={len(np.unique(groups_d))}')

counts, evars, cumvars, maxs = [], [], [], []

for seed in SEEDS:
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for idx_tr, idx_te in sgkf.split(X_d, y_d, groups_d):
        assert len(set(groups_d[idx_tr]) & set(groups_d[idx_te])) == 0
        # FPCA se ajusta directamente sobre X_chol del training fold.
        # sklearn.PCA centra internamente; NO se estandarizan los coeficientes
        # antes de estimar las componentes funcionales.
        X_tr_fpca_input = X_d[idx_tr]

        nc_fold, ev_fold, cum_fold = [], [], []
        for ch in range(n_ch_d):
            p = PCA().fit(X_tr_fpca_input[:, ch, :])
            cum = np.cumsum(p.explained_variance_ratio_)
            n_c = int(np.searchsorted(cum, FPCA_VARIANCE_THRESHOLD) + 1)
            n_c = min(max(n_c, 1), n_basis_d)
            nc_fold.append(n_c)
            ev_fold.append(float(cum[n_c - 1]))
            cum_fold.append(cum)
        counts.append(nc_fold)
        evars.append(ev_fold)
        cumvars.append(cum_fold)
        maxs.append(max(nc_fold))

C   = np.array(counts)      # (n_particiones, n_canales)
E   = np.array(evars)
CUM = np.array(cumvars)     # (n_particiones, n_canales, n_basis)
M   = np.array(maxs)

# ---- Tabla por canal ----
tabla_fpca = pd.DataFrame({
    'Canal': ch_names,
    'Componentes (media)': C.mean(axis=0).round(2),
    'DE': C.std(axis=0, ddof=1).round(2),
    'Minimo': C.min(axis=0),
    'Maximo': C.max(axis=0),
    'Varianza explicada (%)': (E.mean(axis=0) * 100).round(1),
})
print('\n' + tabla_fpca.to_string(index=False))

# ---- Resumen de dimension ----
dim_entrada  = n_ch_d * n_basis_d
dim_relleno  = n_ch_d * M.mean()
dim_efectiva = C.sum(axis=1).mean()

print(f'\nParticiones evaluadas      : {C.shape[0]}')
print(f'Promedio global por canal  : {C.mean():.2f} +/- {C.std(ddof=1):.2f}')
print(f'Rango global               : {C.min()} a {C.max()}')
print(f'n_comp_max por particion   : {M.mean():.2f} +/- {M.std(ddof=1):.2f}')
print(f'\nDimension de entrada       : {n_ch_d} x {n_basis_d} = {dim_entrada}')
print(f'Dimension tras FPCA (relleno, promedio): {dim_relleno:.1f} variables'
      f'   (n_comp_max medio={M.mean():.2f}; reduccion {100*(1-dim_relleno/dim_entrada):.1f}%)')
print(f'Rango de dimension con relleno          : {int(n_ch_d*M.min())} a {int(n_ch_d*M.max())} variables')
print(f'Componentes efectivas      : {dim_efectiva:.1f} +/- {C.sum(axis=1).std(ddof=1):.1f}'
      f'   (reduccion {100*(1-dim_efectiva/dim_entrada):.1f}%)')
print('\nNota: la matriz se rellena con ceros hasta n_comp_max para que todos los')
print('canales tengan la misma dimension dentro de cada fold; por eso la dimension')
print('de entrada puede variar entre folds y no debe resumirse como una multiplicacion')
print('con un numero de componentes redondeado.')
print('La suma de componentes efectivas se reporta por separado.')

# ---- Guardar resultados del diagnóstico ----
if GUARDAR:
    tabla_fpca.to_csv(RESULTS_DIR / 'tabla_fpca_componentes.csv', index=False)
    np.savez_compressed(RESULTS_DIR / 'fpca_diagnostico.npz',
                        counts=C, evar=E, cumvar=CUM, maxs=M,
                        channels=np.array(ch_names))
    print('Tabla:', RESULTS_DIR / 'tabla_fpca_componentes.csv')


## 7. Diagnóstico FPCA del Enfoque B

Se cuantifica el número de componentes retenidos para theta y alfa por canal en las mismas
15 particiones externas. La figura de componentes retenidos por canal y banda se genera a
partir de estas estimaciones específicas de cada partición.

In [ ]:
# ============================================================================
# DIAGNOSTICO FPCA — ENFOQUE B, POR BANDA Y CANAL
# ============================================================================
ENFOQUE_DIAG_B = 'B'
GUARDAR_B = True

X_b, y_b, groups_b, _, extra_b = get_representation_data(
    MAIN_EXPERIMENT, ENFOQUE_DIAG_B, 'X_chol'
)
bands_b = [str(b) for b in extra_b['bands']]
n_basis_per_band_b = [int(v) for v in extra_b['n_basis_per_band']]
n_ch_b = X_b.shape[1]
ch_names_b = CH_NAMES if len(CH_NAMES) == n_ch_b else [f'ch{i}' for i in range(n_ch_b)]
band_starts_b = np.cumsum([0] + n_basis_per_band_b)

print(
    f'Enfoque B: X={X_b.shape} | sujetos={len(np.unique(groups_b))} | '
    f'bandas={bands_b} | bases/banda={n_basis_per_band_b}'
)

records_b = []
dim_padded_b = []
dim_effective_b = []

for seed in SEEDS:
    sgkf = StratifiedGroupKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=seed
    )
    for fold_idx, (idx_tr, idx_te) in enumerate(
        sgkf.split(X_b, y_b, groups_b)
    ):
        assert len(set(groups_b[idx_tr]) & set(groups_b[idx_te])) == 0
        X_tr_b = X_b[idx_tr]

        max_per_band = []
        effective_fold = 0
        for b_idx, band in enumerate(bands_b):
            start = band_starts_b[b_idx]
            end = band_starts_b[b_idx + 1]
            n_basis_band = end - start
            ncomp_band = []

            for ch_idx, ch_name in enumerate(ch_names_b):
                pca = PCA().fit(X_tr_b[:, ch_idx, start:end])
                cum = np.cumsum(pca.explained_variance_ratio_)
                n_comp = int(
                    np.searchsorted(cum, FPCA_VARIANCE_THRESHOLD) + 1
                )
                n_comp = min(max(n_comp, 1), n_basis_band)
                explained = float(cum[n_comp - 1])
                ncomp_band.append(n_comp)
                effective_fold += n_comp
                records_b.append({
                    'seed': seed,
                    'fold': fold_idx,
                    'band': band,
                    'channel': ch_name,
                    'n_components': n_comp,
                    'explained_variance': explained,
                })

            max_per_band.append(max(ncomp_band))

        # fpca_per_band_per_channel rellena cada banda hasta su maximo por canal.
        dim_padded_b.append(n_ch_b * sum(max_per_band))
        dim_effective_b.append(effective_fold)

fpca_b_long = pd.DataFrame(records_b)

summary_b = (
    fpca_b_long
    .groupby(['band', 'channel'], as_index=False)
    .agg(
        componentes_media=('n_components', 'mean'),
        componentes_de=('n_components', 'std'),
        minimo=('n_components', 'min'),
        maximo=('n_components', 'max'),
        varianza_explicada_media=('explained_variance', 'mean'),
    )
)
summary_b['componentes_media'] = summary_b['componentes_media'].round(2)
summary_b['componentes_de'] = summary_b['componentes_de'].fillna(0.0).round(2)
summary_b['varianza_explicada_pct'] = (
    100 * summary_b['varianza_explicada_media']
).round(1)
summary_b = summary_b.drop(columns='varianza_explicada_media')

for band in bands_b:
    print(f'\n--- Banda {band} ---')
    print(
        summary_b[summary_b['band'] == band]
        .drop(columns='band')
        .to_string(index=False)
    )

input_dim_b = int(n_ch_b * sum(n_basis_per_band_b))
dim_padded_b = np.asarray(dim_padded_b, dtype=float)
dim_effective_b = np.asarray(dim_effective_b, dtype=float)

print(f'\nParticiones evaluadas                 : {len(dim_padded_b)}')
print(f'Dimension de entrada B                : {input_dim_b}')
print(
    f'Dimension FPCA con relleno            : '
    f'{dim_padded_b.mean():.1f} +/- {dim_padded_b.std(ddof=1):.1f} '
    f'(rango {int(dim_padded_b.min())}-{int(dim_padded_b.max())})'
)
print(
    f'Componentes efectivas B               : '
    f'{dim_effective_b.mean():.1f} +/- {dim_effective_b.std(ddof=1):.1f}'
)
print(
    f'Reduccion media (dimension con relleno): '
    f'{100*(1-dim_padded_b.mean()/input_dim_b):.1f}%'
)

# Figura: número medio de componentes FPCA retenidos por canal y banda.
BAND_DISPLAY_B = {'theta': 'theta', 'alpha': 'alfa'}
fig, axes = plt.subplots(
    1, len(bands_b), figsize=(6.2 * len(bands_b), 4.2), sharey=True
)
if len(bands_b) == 1:
    axes = [axes]

for ax, band in zip(axes, bands_b):
    sb = (
        summary_b[summary_b['band'] == band]
        .set_index('channel')
        .loc[ch_names_b]
    )
    x = np.arange(len(ch_names_b))
    ax.errorbar(
        x,
        sb['componentes_media'].to_numpy(),
        yerr=sb['componentes_de'].to_numpy(),
        marker='o',
        capsize=3,
        label='Media ± DE',
    )
    ax.set_title(BAND_DISPLAY_B.get(band, band))
    ax.set_xticks(x)
    ax.set_xticklabels(ch_names_b, rotation=45)
    ax.set_xlabel('Canal')
    ax.grid(alpha=0.25, lw=0.5)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('Componentes FPCA retenidos')
fig.tight_layout()

if GUARDAR_B:
    summary_b.to_csv(
        RESULTS_DIR / 'tabla_fpca_componentes_B.csv', index=False
    )
    fpca_b_long.to_csv(
        RESULTS_DIR / 'fpca_diagnostico_B_particiones.csv', index=False
    )
    np.savez_compressed(
        RESULTS_DIR / 'fpca_diagnostico_B.npz',
        dim_padded=dim_padded_b,
        dim_effective=dim_effective_b,
        bands=np.asarray(bands_b),
        channels=np.asarray(ch_names_b),
        n_basis_per_band=np.asarray(n_basis_per_band_b),
    )
    fig_fpca_B_path = FIG_DIR / 'componentes_FPCA_B.png'
    fig.savefig(fig_fpca_B_path, dpi=220, bbox_inches='tight')
    print('\nTabla:', RESULTS_DIR / 'tabla_fpca_componentes_B.csv')
    print('Figura guardada:', fig_fpca_B_path)
plt.show()


## 8. Arquitecturas de aprendizaje profundo


In [ ]:
class MLP(nn.Module):
    def __init__(self, n_channels, n_basis, hidden=(128, 64), dropout=0.3):
        super().__init__()
        in_dim = n_channels * n_basis
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                        nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x.flatten(start_dim=1)).squeeze(-1)


class TransformerCh(nn.Module):
    def __init__(self, n_channels, n_basis, d_model=64, nhead=4,
                  n_layers=2, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(n_basis, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, n_channels, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc   = nn.Linear(d_model, 1)
    def forward(self, x):
        h = self.proj(x) + self.pos_emb
        h = self.encoder(h)
        h = self.norm(h).mean(dim=1)
        return self.fc(h).squeeze(-1)


class FDNN_Direct(nn.Module):
    def __init__(self, n_channels, n_basis, hidden_ch=32,
                  hidden_basis=16, hidden=64, dropout=0.3):
        super().__init__()
        self.channel_proj = nn.Conv1d(n_channels, n_channels * 2,
                                       kernel_size=1, groups=n_channels)
        self.channel_mix  = nn.Conv1d(n_channels * 2, hidden_ch, kernel_size=1)
        self.basis_proj   = nn.Linear(n_basis, hidden_basis)
        self.bn = nn.BatchNorm1d(hidden_ch)
        self.fc1 = nn.Linear(hidden_ch * hidden_basis, hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, x):
        h = F.gelu(self.channel_proj(x))
        h = F.gelu(self.bn(self.channel_mix(h)))
        h = F.gelu(self.basis_proj(h))
        h = h.flatten(start_dim=1)
        h = self.dropout(F.gelu(self.fc1(h)))
        return self.fc2(h).squeeze(-1)


class FNN(nn.Module):
    def __init__(self, n_channels, n_basis, n_functional=16,
                  hidden=64, dropout=0.3):
        super().__init__()
        self.functional_weights = nn.Parameter(
            torch.randn(n_functional, n_channels, n_basis) * (1.0 / np.sqrt(n_basis))
        )
        self.bn = nn.BatchNorm1d(n_functional)
        self.fc1 = nn.Linear(n_functional, hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, x):
        h = torch.einsum('bcn,kcn->bk', x, self.functional_weights)
        h = F.gelu(self.bn(h))
        h = self.dropout(F.gelu(self.fc1(h)))
        return self.fc2(h).squeeze(-1)


class AdaFNN(nn.Module):
    def __init__(self, n_channels, n_basis, n_adaptive=16,
                  hidden=64, dropout=0.3):
        super().__init__()
        self.adaptive_basis = nn.Parameter(
            torch.randn(n_adaptive, n_basis) * (1.0 / np.sqrt(n_basis))
        )
        self.channel_weight = nn.Parameter(
            torch.randn(n_channels, n_adaptive) * (1.0 / np.sqrt(n_channels))
        )
        self.bn  = nn.BatchNorm1d(n_adaptive)
        self.fc1 = nn.Linear(n_adaptive, hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, x):
        h = torch.einsum('bcn,kn->bck', x, self.adaptive_basis)
        h = torch.einsum('bck,ck->bk', h, self.channel_weight)
        h = F.gelu(self.bn(h))
        h = self.dropout(F.gelu(self.fc1(h)))
        return self.fc2(h).squeeze(-1)


class EEGNet(nn.Module):
    def __init__(self, n_channels, n_basis, F1=8, D=2, F2=16,
                  kernel_basis=None, dropout=0.25):
        super().__init__()
        if kernel_basis is None:
            kernel_basis = max(3, n_basis // 2)
        self.conv1 = nn.Conv2d(1, F1, kernel_size=(1, kernel_basis),
                                padding=(0, kernel_basis // 2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)
        self.depthwise = nn.Conv2d(F1, F1 * D, kernel_size=(n_channels, 1),
                                     groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d((1, 2))
        self.drop1 = nn.Dropout(dropout)
        self.separable_dw = nn.Conv2d(F1 * D, F1 * D,
                                        kernel_size=(1, max(3, kernel_basis // 2)),
                                        padding=(0, max(1, kernel_basis // 4)),
                                        groups=F1 * D, bias=False)
        self.separable_pw = nn.Conv2d(F1 * D, F2, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AdaptiveAvgPool2d((1, 1))
        self.drop2 = nn.Dropout(dropout)
        self.fc = nn.Linear(F2, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        h = self.bn1(self.conv1(x))
        h = F.elu(self.bn2(self.depthwise(h)))
        h = self.drop1(self.pool1(h))
        h = self.separable_pw(self.separable_dw(h))
        h = F.elu(self.bn3(h))
        h = self.drop2(self.pool2(h))
        h = h.flatten(start_dim=1)
        return self.fc(h).squeeze(-1)


class CNN1D(nn.Module):
    """Conv1D sobre el eje funcional (n_basis).

    AdaptiveAvgPool1d asegura una salida valida aun cuando el eje funcional
    es corto (p. ej., ~7 componentes tras FPCA). Se incluye para comparar una
    arquitectura convolucional generica en igualdad de condiciones con el resto
    de modelos. Nota: sobre representaciones reducidas por PCA/FPCA, cuyo eje
    esta ordenado por varianza (no por contiguidad espectral), no se espera que
    la convolucion explote estructura local; su desempeno en ese caso es un
    resultado metodologico (interaccion arquitectura x representacion).
    """
    def __init__(self, n_channels, n_basis, dropout=0.3):
        super().__init__()
        k = 3 if n_basis >= 3 else 1
        pad = k // 2
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, k, padding=pad), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, k, padding=pad), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (batch, n_channels, n_basis) -> Conv1d trata canales como in_channels
        return self.net(x).squeeze(-1)


DL_MODELS = {
    'MLP':          MLP,
    'Transformer':  TransformerCh,
    'FDNN_Direct':  FDNN_Direct,
    'FNN':          FNN,
    'AdaFNN':       AdaFNN,
    'EEGNet':       EEGNet,
    'CNN1D':        CNN1D,
}

ML_MODELS = ['LogReg', 'RandomForest']
ALL_MODELS = [(m, 'ml') for m in ML_MODELS] + [(m, 'dl') for m in DL_MODELS.keys()]
print(f'Modelos: {len(ALL_MODELS)} total')
for n, t in ALL_MODELS:
    print(f'  [{t}] {n}')

## 9. Entrenamiento con parada temprana


In [ ]:
def train_dl(model, X_tr, y_tr, X_val, y_val,
              epochs=EPOCHS, patience=PATIENCE,
              lr=1e-3, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE):
    model = model.to(DEVICE)
    X_tr_t  = torch.from_numpy(X_tr.astype(np.float32))
    y_tr_t  = torch.from_numpy(y_tr.astype(np.float32))
    X_val_t = torch.from_numpy(X_val.astype(np.float32)).to(DEVICE)
    y_val_t = torch.from_numpy(y_val.astype(np.float32)).to(DEVICE)

    n_pos = float((y_tr == 1).sum()); n_neg = float((y_tr == 0).sum())
    pos_w = torch.tensor([max(n_neg / max(n_pos, 1.0), 0.5)]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    drop_last = len(X_tr_t) > batch_size
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size,
                        shuffle=True, drop_last=drop_last)

    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    best_val = float('inf'); counter = 0
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            counter = 0
        else:
            counter += 1
            if counter >= patience: break
    model.load_state_dict(best_state)
    return model


def predict_proba_dl(model, X):
    model.eval()
    X_t = torch.from_numpy(X.astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(X_t)).cpu().numpy()

## 10. Evaluación de una configuración


In [ ]:
def _summarize(records):
    df = pd.DataFrame(records)
    s = {}
    for k in ['balanced_acc', 'auc', 'sensitivity', 'specificity', 'f1']:
        s[f'{k}_mean'] = float(df[k].mean())
        s[f'{k}_std'] = float(df[k].std())
    return s


def evaluate_config(
    model_name, model_type, approach, repr_key,
    experiment=MAIN_EXPERIMENT,
    lr=DEFAULT_HP['lr'], dropout=DEFAULT_HP['dropout'],
    seeds=SEEDS, n_folds=N_FOLDS,
):
    """Evalua una configuracion y devuelve metricas a dos niveles.

    Para DL + A_epoca, el split interno por sujeto se realiza ANTES de ajustar
    FPCA/StandardScaler. De esta forma, la validacion interna no participa en
    la estimacion de la representacion usada para early stopping.
    """
    X, y, groups, needs_fpca, extra = get_representation_data(
        experiment, approach, repr_key,
    )
    is_per_epoch = (approach == 'A_epoca')

    rec_epoca, rec_sujeto = [], []
    for seed in seeds:
        set_seed(seed)
        sgkf = StratifiedGroupKFold(
            n_splits=n_folds, shuffle=True, random_state=seed,
        )
        for fold_idx, (idx_tr, idx_te) in enumerate(sgkf.split(X, y, groups)):
            assert len(set(groups[idx_tr]) & set(groups[idx_te])) == 0

            X_tr, X_te = X[idx_tr], X[idx_te]
            y_tr, y_te = y[idx_tr], y[idx_te]
            g_tr, g_te = groups[idx_tr], groups[idx_te]

            if model_type == 'dl' and approach == 'A_epoca':
                # Correccion estricta: split interno -> fit FPCA/scaler en inner-train.
                (
                    X_dl_tr,
                    X_dl_val,
                    X_te_s,
                    y_dl_tr,
                    y_dl_val,
                    _,
                ) = prepare_aepoca_dl_inner_data(
                    X_tr,
                    X_te,
                    y_tr,
                    g_tr,
                    repr_key,
                    seed,
                    fold_idx,
                )

                n_channels, n_basis = X_dl_tr.shape[1], X_dl_tr.shape[2]
                cls = DL_MODELS[model_name]
                try:
                    model = cls(
                        n_channels=n_channels,
                        n_basis=n_basis,
                        dropout=dropout,
                    )
                except TypeError:
                    model = cls(n_channels=n_channels, n_basis=n_basis)
                model = train_dl(
                    model,
                    X_dl_tr,
                    y_dl_tr,
                    X_dl_val,
                    y_dl_val,
                    lr=lr,
                )
                y_proba = predict_proba_dl(model, X_te_s)

            else:
                # Pipeline previo para ML y para los escenarios por sujeto A/B.
                if needs_fpca:
                    if approach == 'B':
                        X_tr_repr, X_te_repr, _ = fpca_per_band_per_channel(
                            X_tr,
                            X_te,
                            extra['n_basis_per_band'],
                        )
                    else:
                        X_tr_repr, X_te_repr, _ = fpca_per_channel(X_tr, X_te)
                    X_tr_s, X_te_s = standardize_per_fold(
                        X_tr_repr,
                        X_te_repr,
                    )
                else:
                    X_tr_s, X_te_s = standardize_per_fold(X_tr, X_te)

                n_channels, n_basis = X_tr_s.shape[1], X_tr_s.shape[2]

                if model_type == 'ml':
                    if model_name == 'LogReg':
                        clf = LogisticRegression(
                            C=1.0,
                            class_weight='balanced',
                            max_iter=2000,
                            random_state=RANDOM_STATE,
                        )
                    else:
                        clf = RandomForestClassifier(
                            n_estimators=200,
                            max_depth=10,
                            class_weight='balanced',
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                        )
                    clf.fit(X_tr_s.reshape(len(X_tr_s), -1), y_tr)
                    y_proba = clf.predict_proba(
                        X_te_s.reshape(len(X_te_s), -1)
                    )[:, 1]
                else:
                    tr_in, val_in = make_inner_train_val_split(
                        y_tr, g_tr, approach, seed, fold_idx,
                    )
                    cls = DL_MODELS[model_name]
                    try:
                        model = cls(
                            n_channels=n_channels,
                            n_basis=n_basis,
                            dropout=dropout,
                        )
                    except TypeError:
                        model = cls(n_channels=n_channels, n_basis=n_basis)
                    model = train_dl(
                        model,
                        X_tr_s[tr_in],
                        y_tr[tr_in],
                        X_tr_s[val_in],
                        y_tr[val_in],
                        lr=lr,
                    )
                    y_proba = predict_proba_dl(model, X_te_s)

            # Nivel epoca (prediccion cruda, sin agregar).
            m_ep = compute_metrics(y_te, y_proba)
            m_ep.update({'seed': seed, 'fold': fold_idx})
            rec_epoca.append(m_ep)

            # Nivel sujeto.
            if is_per_epoch:
                y_te_su, y_proba_su = aggregate_by_subject(y_proba, y_te, g_te)
                m_su = compute_metrics(y_te_su, y_proba_su)
            else:
                m_su = compute_metrics(y_te, y_proba)
            m_su.update({'seed': seed, 'fold': fold_idx})
            rec_sujeto.append(m_su)

    out = {'sujeto': _summarize(rec_sujeto)}
    out['epoca'] = _summarize(rec_epoca) if is_per_epoch else None
    return out


### 10.1 Partición interna por sujeto en A2

Para los modelos profundos de A2, la validación interna utilizada por la parada temprana se construye manteniendo agrupadas todas las épocas de un mismo sujeto.


In [ ]:
# ============================================================================
# A_epoca — split interno agrupado por sujeto para early stopping
# ============================================================================
from sklearn.model_selection import StratifiedShuffleSplit


def make_inner_train_val_split(
    y_train: np.ndarray,
    groups_train: np.ndarray,
    approach: str,
    seed: int,
    fold_idx: int,
    val_fraction: float = 0.20,
) -> Tuple[np.ndarray, np.ndarray]:
    """Crea train/validacion interna sin compartir sujetos en A_epoca.

    Para A y B (una fila por sujeto), conserva el split aleatorio previo.
    Para A_epoca, divide primero los sujetos de forma estratificada y luego
    recupera todas las epocas de cada sujeto seleccionado.
    """
    if approach != 'A_epoca':
        rng_split = np.random.default_rng(seed + fold_idx)
        permutation = rng_split.permutation(len(y_train))
        n_val = max(4, int(val_fraction * len(permutation)))
        cut = len(permutation) - n_val
        return permutation[:cut], permutation[cut:]

    groups_train = np.asarray(groups_train)
    y_train = np.asarray(y_train)
    subjects = np.unique(groups_train)
    subject_labels = []
    for subject in subjects:
        labels = np.unique(y_train[groups_train == subject])
        if len(labels) != 1:
            raise ValueError(
                f'Etiquetas inconsistentes para el sujeto {subject}: {labels}'
            )
        subject_labels.append(int(labels[0]))
    subject_labels = np.asarray(subject_labels)

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_fraction,
        random_state=seed + fold_idx,
    )
    idx_sub_tr, idx_sub_val = next(
        splitter.split(np.zeros((len(subjects), 1)), subject_labels)
    )
    subjects_tr = subjects[idx_sub_tr]
    subjects_val = subjects[idx_sub_val]

    idx_tr = np.flatnonzero(np.isin(groups_train, subjects_tr))
    idx_val = np.flatnonzero(np.isin(groups_train, subjects_val))

    overlap = set(groups_train[idx_tr]) & set(groups_train[idx_val])
    if overlap:
        raise AssertionError(
            f'Fuga de sujetos en validacion interna A_epoca: {sorted(overlap)}'
        )
    return idx_tr, idx_val


print(
    'A_epoca: early stopping usara validacion interna estratificada y agrupada '
    'por sujeto (sin compartir epocas del mismo sujeto entre train y validacion).'
)

### 10.2 Ajuste de transformaciones en A2

En A2, las transformaciones dependientes de los datos se ajustan únicamente con el subconjunto interno de entrenamiento y posteriormente se aplican a validación y prueba.


In [ ]:
# ============================================================================
# A_epoca — preprocesamiento estricto para DL: fit SOLO con inner-train
# ============================================================================
def prepare_aepoca_dl_inner_data(
    X_outer_train: np.ndarray,
    X_outer_test: np.ndarray,
    y_outer_train: np.ndarray,
    groups_outer_train: np.ndarray,
    repr_key: str,
    seed: int,
    fold_idx: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict]:
    """Prepara inner-train, inner-val y outer-test sin fuga interna en A_epoca.

    Orden para X_chol_fpca:
        inner-train X_chol -> FPCA(fit inner-train) -> StandardScaler(fit inner-train scores)
        inner-val/test      -> transform con esos mismos objetos

    Orden para X_raw/X_chol:
        inner-train -> StandardScaler(fit inner-train)
        inner-val/test -> transform con el mismo scaler

    La funcion reutiliza los helpers existentes concatenando val+test SOLO como
    conjunto de transformacion; nunca participan en fit de PCA ni del scaler.
    """
    idx_inner_train, idx_inner_val = make_inner_train_val_split(
        y_outer_train,
        groups_outer_train,
        'A_epoca',
        seed,
        fold_idx,
    )

    X_inner_raw = X_outer_train[idx_inner_train]
    X_val_raw = X_outer_train[idx_inner_val]
    X_eval_raw = np.concatenate([X_val_raw, X_outer_test], axis=0)
    n_val = len(X_val_raw)

    info: Dict = {
        'inner_train_indices': idx_inner_train,
        'inner_val_indices': idx_inner_val,
    }

    if repr_key == 'X_chol_fpca':
        X_inner_repr, X_eval_repr, fpca_info = fpca_per_channel(
            X_inner_raw,
            X_eval_raw,
        )
        info['fpca'] = fpca_info
    elif repr_key in ('X_raw', 'X_chol'):
        X_inner_repr = X_inner_raw
        X_eval_repr = X_eval_raw
    else:
        raise ValueError(f'Representacion no reconocida para A_epoca: {repr_key}')

    # El scaler se ajusta EXCLUSIVAMENTE con inner-train.
    X_inner_scaled, X_eval_scaled = standardize_per_fold(
        X_inner_repr,
        X_eval_repr,
    )

    X_val_scaled = X_eval_scaled[:n_val]
    X_test_scaled = X_eval_scaled[n_val:]
    y_inner_train = y_outer_train[idx_inner_train]
    y_inner_val = y_outer_train[idx_inner_val]

    train_subjects = set(groups_outer_train[idx_inner_train])
    val_subjects = set(groups_outer_train[idx_inner_val])
    if train_subjects & val_subjects:
        raise AssertionError('Fuga de sujetos entre inner-train e inner-val.')

    return (
        X_inner_scaled,
        X_val_scaled,
        X_test_scaled,
        y_inner_train,
        y_inner_val,
        info,
    )


print(
    'A_epoca DL: FPCA/StandardScaler se ajustaran solo con inner-train; '
    'inner-val y outer-test seran exclusivamente transformados.'
)


## 11. Línea base escalar sin ADF

Se reconstruye la línea base convencional por sujeto a partir de las épocas preprocesadas, utilizando el mismo conjunto final de 48 sujetos.


In [ ]:
# ==== Escenario N0: caracteristicas escalares del promedio de epocas ====
from scipy.signal import welch
from scipy.integrate import simpson

MANIFEST_N0 = FINAL_ROOT / '01_preprocesamiento' / 'epochs_manifest.csv'
LABELS_N0   = {'CTL': 0, 'PD_OFF': 1}
BANDS_N0    = {'delta':(1.0,4.0),'theta':(4.0,8.0),'pre_alpha':(5.5,8.0),
               'alpha':(8.0,13.0),'beta':(13.0,30.0)}
PSD_FMIN_N0, PSD_FMAX_N0, NORM_FMAX_N0 = 1.0, 40.0, 30.0
WELCH_SEC_N0, WELCH_OVL_N0 = 2.0, 0.5   # identicos a los cuadernos 02A/02B


def _psd_avg_subject(data_uv, sfreq):
    """PSD promedio (Welch, sobre epocas) por sujeto. -> (psd_avg (n_ch,n_freqs), freqs)."""
    n_ep, n_ch, n_t = data_uv.shape
    nperseg = min(int(sfreq * WELCH_SEC_N0), n_t); nov = int(nperseg * WELCH_OVL_N0)
    psd = np.stack([welch(data_uv[i], fs=sfreq, nperseg=nperseg, noverlap=nov, axis=-1)[1]
                    for i in range(n_ep)], axis=0)
    f = welch(data_uv[0], fs=sfreq, nperseg=nperseg, noverlap=nov, axis=-1)[0]
    m = (f >= PSD_FMIN_N0) & (f <= PSD_FMAX_N0)
    return psd[:, :, m].mean(axis=0), f[m]   # <- promedio sobre epocas


def _scalar_features(psd_avg, freqs):
    """Escalares clasicos por canal: 5 pot. rel. + frec. dominante + ratio alpha/theta."""
    df = freqs[1] - freqs[0]; tm = (freqs >= PSD_FMIN_N0) & (freqs <= NORM_FMAX_N0)
    tot = simpson(psd_avg[..., tm], dx=df, axis=-1); tot = np.where(tot < 1e-20, 1e-20, tot)
    feats, bp = [], {}
    for name, (lo, hi) in BANDS_N0.items():
        mm = (freqs >= lo) & (freqs <= hi)
        p = simpson(psd_avg[..., mm], dx=df, axis=-1) / tot
        feats.append(np.log10(p + 1e-20)); bp[name] = p
    fm = (freqs >= PSD_FMIN_N0) & (freqs <= NORM_FMAX_N0); fr = freqs[fm]
    feats.append(fr[psd_avg[..., fm].argmax(axis=-1)])                     # frecuencia dominante
    feats.append(np.log10(bp['alpha'] / (bp['theta'] + 1e-20) + 1e-20))   # ratio alpha/theta
    return np.stack(feats, axis=-1)   # (n_ch, n_feats)


import time


def _remount_drive():
    """Remonta Drive si el endpoint se cayo (Errno 107)."""
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception:
        pass
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    time.sleep(3)


def _load_npy_robust(path, max_retries: int = 3):
    """Carga un .npy con reintentos ante caidas del montaje de Drive."""
    for attempt in range(max_retries):
        try:
            return np.load(path).astype(np.float64)
        except OSError as e:
            if getattr(e, 'errno', None) == 107 or 'Transport endpoint' in str(e):
                print(f'  Drive caido (intento {attempt+1}/{max_retries}), remontando...')
                _remount_drive()
            else:
                raise
    raise OSError(f'No se pudo leer {path} tras {max_retries} intentos (Drive inestable)')


# --- Construir datos N0 (un vector escalar por sujeto) ---
man = pd.read_csv(MANIFEST_N0)
man['error'] = man.get('error', '').fillna('').astype(str)
man = man[(man['error'] == '') & (man['n_epocas_conservadas'] >= 6)
          & (man['condition'].isin(LABELS_N0))].copy()
man['label'] = man['condition'].map(LABELS_N0)
man = man.sort_values(['condition', 'subject_id']).reset_index(drop=True)

X_list, y_N0, g_N0 = [], [], []
for _, r in man.iterrows():
    data = _load_npy_robust(r['file']); sf = float(r['sfreq'])
    psd_avg, freqs = _psd_avg_subject(data, sf)
    X_list.append(_scalar_features(psd_avg, freqs))
    y_N0.append(r['label']); g_N0.append(r['subject_id'])
X_N0 = np.stack(X_list, axis=0)             # (n_subj, n_ch, n_feats)
y_N0 = np.asarray(y_N0); g_N0 = np.asarray(g_N0)

print(f'N0 (sin FDA): X={X_N0.shape}  sujetos={len(np.unique(g_N0))}  clases={np.bincount(y_N0)}')
# verificacion: mismo numero de sujetos que el dataset A1
n_fda = len(np.unique(all_datasets[MAIN_EXPERIMENT]['A']['groups']))
print(f'Sujetos en FDA (A): {n_fda}  |  coincide con N0: {n_fda == len(np.unique(g_N0))}')


### 11.1 Evaluación de la línea base


In [ ]:
# ==== Evaluador de N0 (mismo esquema de CV; solo modelos compatibles con escalar) ====
def evaluate_N0(model_name, model_type,
                lr=DEFAULT_HP['lr'], dropout=DEFAULT_HP['dropout'],
                seeds=SEEDS, n_folds=N_FOLDS):
    """Evalua el baseline N0 (escalar por sujeto). N0 es por sujeto -> sin agregacion."""
    fold_records = []
    for seed in seeds:
        set_seed(seed)
        sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for fold_idx, (idx_tr, idx_te) in enumerate(sgkf.split(X_N0, y_N0, g_N0)):
            assert len(set(g_N0[idx_tr]) & set(g_N0[idx_te])) == 0
            X_tr, X_te = X_N0[idx_tr], X_N0[idx_te]
            y_tr, y_te = y_N0[idx_tr], y_N0[idx_te]
            X_tr_s, X_te_s = standardize_per_fold(X_tr, X_te)

            if model_type == 'ml':
                if model_name == 'LogReg':
                    clf = LogisticRegression(C=1.0, class_weight='balanced',
                                              max_iter=2000, random_state=RANDOM_STATE)
                else:
                    clf = RandomForestClassifier(n_estimators=200, max_depth=10,
                                                  class_weight='balanced',
                                                  random_state=RANDOM_STATE, n_jobs=-1)
                clf.fit(X_tr_s.reshape(len(X_tr_s), -1), y_tr)
                y_proba = clf.predict_proba(X_te_s.reshape(len(X_te_s), -1))[:, 1]
            else:  # solo MLP entre los DL es compatible con N0
                rng_split = np.random.default_rng(seed + fold_idx)
                tr_idx = rng_split.permutation(len(X_tr_s)); n_val = max(4, int(0.2 * len(tr_idx)))
                cut = len(tr_idx) - n_val; tr_in, val_in = tr_idx[:cut], tr_idx[cut:]
                cls = DL_MODELS[model_name]; nch, nb = X_tr_s.shape[1], X_tr_s.shape[2]
                try: model = cls(n_channels=nch, n_basis=nb, dropout=dropout)
                except TypeError: model = cls(n_channels=nch, n_basis=nb)
                model = train_dl(model, X_tr_s[tr_in], y_tr[tr_in],
                                  X_tr_s[val_in], y_tr[val_in], lr=lr)
                y_proba = predict_proba_dl(model, X_te_s)

            m = compute_metrics(y_te, y_proba)   # N0 es por sujeto
            m.update({'seed': seed, 'fold': fold_idx}); fold_records.append(m)

    df = pd.DataFrame(fold_records); summary = {}
    for k in ['balanced_acc','auc','sensitivity','specificity','f1']:
        summary[f'{k}_mean'] = float(df[k].mean()); summary[f'{k}_std'] = float(df[k].std())
    return summary

### 11.2 Dimensión de entrada y número de parámetros: Cholesky frente a FPCA

Se cuantifica la reducción de dimensión y de parámetros entrenables que introduce la FPCA,
usando el número de componentes observado en las 15 particiones del diagnóstico del Enfoque A.

In [ ]:
# ============================================================================
# CUANTIFICACION DEL AHORRO DE FPCA: dimension y parametros (dinamico)
# Usa el n_comp_max observado en las 15 particiones del diagnostico del Enfoque A.
# No se fija manualmente N_FPCA.
# ============================================================================
N_CH = int(n_ch_d)
N_CHOL = int(n_basis_d)

if 'M' not in globals() or len(M) == 0:
    raise RuntimeError(
        'Ejecuta primero el diagnostico FPCA del Enfoque A (seccion 6).'
    )

fpca_nmax_por_particion = np.asarray(M, dtype=int)


def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)


input_chol = N_CH * N_CHOL
input_fpca = N_CH * fpca_nmax_por_particion
print(f'Dimension de entrada Cholesky: {input_chol}')
print(
    'Dimension de entrada FPCA con relleno: '
    f'{input_fpca.mean():.1f} +/- {input_fpca.std(ddof=1):.1f} '
    f'(rango {input_fpca.min()}-{input_fpca.max()})'
)
print(
    f'Reduccion media de dimension: '
    f'{100*(1-input_fpca.mean()/input_chol):.1f}%\n'
)

filas = []
print(
    f'{"Modelo":<14}{"Chol":>12}{"FPCA media":>14}'
    f'{"FPCA min-max":>18}{"Reduccion":>13}'
)
print('-' * 72)
for nombre, cls in DL_MODELS.items():
    try:
        model_chol = cls(n_channels=N_CH, n_basis=N_CHOL)
    except TypeError:
        model_chol = cls(N_CH, N_CHOL)
    params_chol = contar_parametros(model_chol)

    params_fpca = []
    for n_fpca_fold in fpca_nmax_por_particion:
        try:
            model_fpca = cls(n_channels=N_CH, n_basis=int(n_fpca_fold))
        except TypeError:
            model_fpca = cls(N_CH, int(n_fpca_fold))
        params_fpca.append(contar_parametros(model_fpca))
    params_fpca = np.asarray(params_fpca, dtype=float)
    reduction = 100 * (1 - params_fpca.mean() / params_chol) if params_chol else 0.0

    filas.append({
        'modelo': nombre,
        'params_chol': int(params_chol),
        'params_fpca_mean': float(params_fpca.mean()),
        'params_fpca_std': float(params_fpca.std(ddof=1)),
        'params_fpca_min': int(params_fpca.min()),
        'params_fpca_max': int(params_fpca.max()),
        'reduccion_media_pct': float(reduction),
    })
    print(
        f'{nombre:<14}{params_chol:>12,}{params_fpca.mean():>14,.1f}'
        f'{f"{int(params_fpca.min()):,}-{int(params_fpca.max()):,}":>18}'
        f'{reduction:>12.1f}%'
    )

pd.DataFrame(filas).to_csv(
    RESULTS_DIR / 'params_chol_vs_fpca.csv', index=False
)
print('\nGuardado en params_chol_vs_fpca.csv')


## 12. Evaluación unificada

Se ejecutan las configuraciones A1, A2, B y la línea base bajo el esquema de validación definido anteriormente. Los resultados se guardan incrementalmente para preservar la trazabilidad de la ejecución.


### Ejecución

Esta sección es la más costosa del cuaderno porque evalúa todos los modelos,
representaciones, pliegues y semillas (del orden de 10 minutos con GPU). Las métricas
finales se calculan a nivel de sujeto.

In [ ]:
# ==== Evaluacion unificada — todos los escenarios x todos los modelos ====
# El baseline sin ADF (escalar por sujeto) solo admite modelos que tratan la
# entrada como vector generico o atienden sobre canales; se excluyen los
# convolucionales/funcionales que requieren continuidad en el eje funcional.
N0_COMPATIBLE = {'LogReg', 'RandomForest', 'MLP', 'Transformer'}
BASELINE_NAME = 'Baseline_sin_ADF'

SCENARIOS = ([(BASELINE_NAME, None, BASELINE_NAME)]
             + [('A', r, r) for r in REPRESENTATIONS]
             + [('B', r, r) for r in REPRESENTATIONS]
             + [('A_epoca', r, r) for r in REPRESENTATIONS])

CSV_PATH = RESULTS_DIR / 'tabla_comparativa_completa.csv'
records = []
for scen, repr_key, tag in SCENARIOS:
    for model_name, model_type in ALL_MODELS:
        if scen == BASELINE_NAME and model_name not in N0_COMPATIBLE:
            records.append(dict(scenario=BASELINE_NAME, repr=tag, model=model_name,
                                nivel='sujeto', status='N/A'))
            continue
        try:
            if scen == BASELINE_NAME:
                res = {'sujeto': evaluate_N0(model_name, model_type,
                                             lr=DEFAULT_HP['lr'], dropout=DEFAULT_HP['dropout']),
                       'epoca': None}
            else:
                res = evaluate_config(model_name, model_type, scen, repr_key,
                                      lr=DEFAULT_HP['lr'], dropout=DEFAULT_HP['dropout'])
            for nivel in ['epoca', 'sujeto']:
                if res.get(nivel) is None:
                    continue
                records.append(dict(scenario=scen, repr=tag, model=model_name,
                                    nivel=nivel, status='ok', **res[nivel]))
            su = res['sujeto']
            print(f"{scen:16s} {tag:12s} {model_name:12s} [sujeto] "
                  f"BalAcc={su['balanced_acc_mean']:.3f}  AUC={su['auc_mean']:.3f}")
        except Exception as e:
            records.append(dict(scenario=scen, repr=tag, model=model_name,
                                nivel='sujeto', status=f'error: {e}'))
            print(f"{scen:16s} {tag:12s} {model_name:12s} ERROR: {e}")
        pd.DataFrame(records).to_csv(CSV_PATH, index=False)   # guardado incremental

DF_ALL = pd.DataFrame(records)
print(f'\nGuardado (incremental) en: {CSV_PATH}')


## 13. Tablas comparativas de resultados


In [ ]:
# ==== Tablas: media +/- DE, por nivel (epoca y sujeto) ====
ok = DF_ALL[DF_ALL['status'] == 'ok'].copy()
METRICS = [('balanced_acc', 'BalAcc'), ('auc', 'AUC'), ('sensitivity', 'Sens'),
           ('specificity', 'Spec'), ('f1', 'F1')]


def _fmt(row, m):
    return f"{row[f'{m}_mean']:.3f} ± {row[f'{m}_std']:.3f}"


for nivel in ['sujeto', 'epoca']:
    sub = ok[ok['nivel'] == nivel].copy()
    if len(sub) == 0:
        continue
    sub['config'] = sub['scenario'] + ' / ' + sub['repr']
    rows = []
    for _, r in sub.iterrows():
        row = {'config': r['config'], 'modelo': r['model']}
        for m, lbl in METRICS:
            row[lbl] = _fmt(r, m)
        rows.append(row)
    tab = pd.DataFrame(rows).set_index(['config', 'modelo'])
    print(f'\n===== NIVEL {nivel.upper()} (media ± DE sobre folds y semillas) =====')
    if nivel == 'epoca':
        print('(Solo A_epoca: cada epoca es una muestra. A base, B y baseline son por sujeto.)')
    print(tab.to_string())
    tab.to_csv(RESULTS_DIR / f'tabla_{nivel}_media_std.csv')

# Pivote rapido (media) para BalAcc y AUC a nivel sujeto
piv_sub = ok[ok['nivel'] == 'sujeto'].copy()
piv_sub['cond'] = piv_sub['scenario'] + ' / ' + piv_sub['repr']
for metric, lbl in [('balanced_acc', 'BalAcc'), ('auc', 'AUC')]:
    piv = piv_sub.pivot_table(index='cond', columns='model', values=f'{metric}_mean')
    print(f'\n=== Pivote {lbl} (media, nivel sujeto) ===')
    print(piv.round(3).to_string())

na = DF_ALL[DF_ALL['status'] == 'N/A']
if len(na):
    print(f"\nN/A ({len(na)} celdas) — modelos no aplicables al baseline escalar: "
          f"{sorted(na['model'].unique())}")
err = DF_ALL[DF_ALL['status'].astype(str).str.startswith('error')]
if len(err):
    print(f"\nERRORES ({len(err)}): revisar\n",
          err[['scenario', 'repr', 'model', 'status']].to_string())


## 14. Figuras y diagnósticos del modelado

En este cuaderno se generan únicamente las figuras que dependen del proceso de validación o del entrenamiento de los modelos: componentes FPCA del Enfoque B, matrices de confusión, curvas de entrenamiento y curva de aprendizaje por número de sujetos.

Las figuras descriptivas y de reconstrucción de los Enfoques A y B se generan exclusivamente en los cuadernos `02A` y `02B` para evitar duplicación.


### 14.1 Configuración de las figuras de modelado


In [ ]:
# Configuración de figuras que dependen del modelado.
# No se regeneran ni copian las figuras descriptivas de 01, 02A o 02B.

MODEL_FIG_DIR = FIG_DIR
FIGURE_DATA_DIR = RESULTS_DIR / 'datos_figuras'
FIGURE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Los diagnósticos de las figuras se recalculan siempre; no se reutilizan archivos intermedios previos.
FORCE_REGENERATE_FIGURES = True
FIGURE_CACHE_TAG = 'modelado'
HISTORY_SEEDS = list(SEEDS)
HISTORY_FOLDS = list(range(N_FOLDS))

# Curva de aprendizaje por número de sujetos independientes.
LEARNING_SUBJECT_SIZES = [12, 18, 24, 30, 36, 38]
LEARNING_REPEATS = 5
LEARNING_TEST_PER_CLASS = 5

print('Carpeta de figuras de modelado:', MODEL_FIG_DIR)

REPR_DISPLAY = {
    'X_raw': 'B-spline',
    'X_chol': 'Cholesky',
    'X_chol_fpca': 'Cholesky + FPCA',
}


def _best_result_row(
    scenario: str,
    deep_only: bool = False,
    per_model: bool = False,
) -> pd.DataFrame:
    """Filtra resultados a nivel de sujeto y ordena por exactitud balanceada."""
    subset = DF_ALL[
        (DF_ALL['status'] == 'ok')
        & (DF_ALL['nivel'] == 'sujeto')
        & (DF_ALL['scenario'] == scenario)
    ].copy()
    if deep_only:
        subset = subset[subset['model'].isin(DL_MODELS.keys())]
    subset = subset.sort_values(
        ['balanced_acc_mean', 'auc_mean'], ascending=[False, False]
    )
    if per_model:
        subset = subset.drop_duplicates(subset=['model'], keep='first')
    if subset.empty:
        raise RuntimeError(f'No hay resultados disponibles para {scenario}.')
    return subset


def _result_to_config(row: pd.Series, title_prefix: str = '') -> dict:
    repr_label = REPR_DISPLAY.get(str(row['repr']), str(row['repr']))
    title = f"{row['model']} + {repr_label}"
    if title_prefix:
        title = f"{title_prefix}{title}"
    return {
        'title': title,
        'label': title,
        'model': str(row['model']),
        'approach': str(row['scenario']),
        'repr': str(row['repr']),
    }


### 14.2 Matrices de confusión

Matrices normalizadas por clase real, promediadas sobre las 15 particiones, para tres
configuraciones seleccionadas automáticamente a partir de los resultados: la mejor configuración
de A1, el mejor modelo profundo de A1 y el mejor modelo profundo de A2.

In [ ]:
# ============================================================================
# HELPERS COMUNES PARA DIAGNOSTICOS Y FIGURAS
# ============================================================================
def _safe_auc(y_true: np.ndarray, y_proba: np.ndarray) -> float:
    """Calcula AUC y devuelve NaN cuando una particion tiene una sola clase."""
    try:
        return float(roc_auc_score(y_true, y_proba))
    except ValueError:
        return np.nan


def _build_dl_model(
    model_name: str,
    n_channels: int,
    n_basis: int,
    dropout: float = DEFAULT_HP['dropout'],
) -> nn.Module:
    """Instancia una arquitectura usando la misma configuracion del pipeline."""
    cls = DL_MODELS[model_name]
    try:
        return cls(n_channels=n_channels, n_basis=n_basis, dropout=dropout)
    except TypeError:
        return cls(n_channels=n_channels, n_basis=n_basis)


def _prepare_fold_data(
    approach: str,
    repr_key: str,
    idx_train: np.ndarray,
    idx_test: np.ndarray,
):
    """Preprocesamiento externo usado por ML y por escenarios A/B por sujeto."""
    X, y, groups, needs_fpca, extra = get_representation_data(
        MAIN_EXPERIMENT, approach, repr_key,
    )
    X_train, X_test = X[idx_train], X[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]
    groups_test = groups[idx_test]

    if needs_fpca:
        if approach == 'B':
            X_train, X_test, _ = fpca_per_band_per_channel(
                X_train,
                X_test,
                extra['n_basis_per_band'],
            )
        else:
            X_train, X_test, _ = fpca_per_channel(X_train, X_test)
        X_train, X_test = standardize_per_fold(X_train, X_test)
    else:
        X_train, X_test = standardize_per_fold(X_train, X_test)

    return X_train, X_test, y_train, y_test, groups_test


def _fit_predict_fold(
    model_name: str,
    approach: str,
    repr_key: str,
    idx_train: np.ndarray,
    idx_test: np.ndarray,
    seed: int,
    fold_idx: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Entrena una configuracion reproduciendo el pipeline de evaluacion."""
    X_all, y_all, groups_all, _, _ = get_representation_data(
        MAIN_EXPERIMENT, approach, repr_key,
    )

    if model_name not in ML_MODELS and approach == 'A_epoca':
        X_outer_train, X_outer_test = X_all[idx_train], X_all[idx_test]
        y_outer_train, y_test = y_all[idx_train], y_all[idx_test]
        groups_outer_train = groups_all[idx_train]
        groups_test = groups_all[idx_test]

        (
            X_train,
            X_val,
            X_test,
            y_train,
            y_val,
            _,
        ) = prepare_aepoca_dl_inner_data(
            X_outer_train,
            X_outer_test,
            y_outer_train,
            groups_outer_train,
            repr_key,
            seed,
            fold_idx,
        )
        model = _build_dl_model(
            model_name,
            n_channels=X_train.shape[1],
            n_basis=X_train.shape[2],
        )
        model = train_dl(
            model,
            X_train,
            y_train,
            X_val,
            y_val,
            lr=DEFAULT_HP['lr'],
        )
        y_proba = predict_proba_dl(model, X_test)
        return y_test, y_proba, groups_test

    X_train, X_test, y_train, y_test, groups_test = _prepare_fold_data(
        approach, repr_key, idx_train, idx_test,
    )

    if model_name in ML_MODELS:
        if model_name == 'LogReg':
            model = LogisticRegression(
                C=1.0,
                class_weight='balanced',
                max_iter=2000,
                random_state=RANDOM_STATE,
            )
        else:
            model = RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        model.fit(X_train.reshape(len(X_train), -1), y_train)
        y_proba = model.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]
    else:
        groups_train = groups_all[idx_train]
        train_inner, val_inner = make_inner_train_val_split(
            y_train, groups_train, approach, seed, fold_idx,
        )
        model = _build_dl_model(
            model_name,
            n_channels=X_train.shape[1],
            n_basis=X_train.shape[2],
        )
        model = train_dl(
            model,
            X_train[train_inner],
            y_train[train_inner],
            X_train[val_inner],
            y_train[val_inner],
            lr=DEFAULT_HP['lr'],
        )
        y_proba = predict_proba_dl(model, X_test)

    return y_test, y_proba, groups_test


def _subject_level_arrays(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    groups: np.ndarray,
    approach: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Agrega probabilidades por sujeto solo para el experimento por epocas."""
    if approach == 'A_epoca':
        return aggregate_by_subject(y_proba, y_true, groups)
    return y_true, y_proba


In [ ]:
# ============================================================================
# FIGURA: MATRICES DE CONFUSION PROMEDIO SOBRE 15 PARTICIONES
# ============================================================================
# Seleccion automatica a partir de los resultados de la evaluacion:
# 1) mejor configuracion global de A por sujeto;
# 2) mejor modelo profundo de A por sujeto;
# 3) mejor modelo profundo de A_epoca a nivel sujeto.
_best_A_global = _best_result_row('A').iloc[0]
_best_A_deep = _best_result_row('A', deep_only=True).iloc[0]
_best_Aep_deep = _best_result_row('A_epoca', deep_only=True).iloc[0]

CONFUSION_CONFIGS = [
    _result_to_config(_best_A_global),
    _result_to_config(_best_A_deep),
    _result_to_config(_best_Aep_deep),
]

print('Configuraciones para matrices de confusion:')
for cfg in CONFUSION_CONFIGS:
    print(f"  - {cfg['approach']} | {cfg['model']} | {cfg['repr']}")


def collect_confusion_diagnostics(config: dict) -> tuple[np.ndarray, pd.DataFrame]:
    """Obtiene una matriz normalizada por fold y las metricas correspondientes."""
    approach = config['approach']
    repr_key = config['repr']
    X, y, groups, _, _ = get_representation_data(
        MAIN_EXPERIMENT, approach, repr_key,
    )

    matrices = []
    rows = []
    for seed in SEEDS:
        set_seed(seed)
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=seed,
        )
        for fold_idx, (idx_train, idx_test) in enumerate(
            splitter.split(X, y, groups)
        ):
            y_test, y_proba, groups_test = _fit_predict_fold(
                config['model'],
                approach,
                repr_key,
                idx_train,
                idx_test,
                seed,
                fold_idx,
            )
            y_subject, p_subject = _subject_level_arrays(
                y_test, y_proba, groups_test, approach,
            )
            y_pred = (p_subject >= 0.5).astype(int)
            cm = confusion_matrix(
                y_subject,
                y_pred,
                labels=[0, 1],
                normalize='true',
            )
            matrices.append(cm)
            metrics = compute_metrics(y_subject, p_subject)
            rows.append({
                'seed': seed,
                'fold': fold_idx,
                'model': config['model'],
                'approach': approach,
                'repr': repr_key,
                **metrics,
            })

    return np.stack(matrices), pd.DataFrame(rows)


confusion_results = {}
all_confusion_rows = []
for config in CONFUSION_CONFIGS:
    slug = f"{config['approach']}_{config['repr']}_{config['model']}"
    cache_npz = FIGURE_DATA_DIR / f'confusion_{slug}.npz'
    cache_csv = FIGURE_DATA_DIR / f'confusion_{slug}.csv'

    if cache_npz.exists() and cache_csv.exists() and not FORCE_REGENERATE_FIGURES:
        with np.load(cache_npz, allow_pickle=False) as saved:
            matrices = saved['matrices']
        metrics_df = pd.read_csv(cache_csv)
    else:
        matrices, metrics_df = collect_confusion_diagnostics(config)
        np.savez_compressed(cache_npz, matrices=matrices)
        metrics_df.to_csv(cache_csv, index=False)

    confusion_results[slug] = {
        'config': config,
        'matrices': matrices,
        'metrics': metrics_df,
    }
    all_confusion_rows.append(metrics_df)

pd.concat(all_confusion_rows, ignore_index=True).to_csv(
    FIGURE_DATA_DIR / 'matrices_confusion_metricas_folds.csv',
    index=False,
)

fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.2), constrained_layout=True)
last_image = None
for ax, result in zip(axes, confusion_results.values()):
    mean_cm = result['matrices'].mean(axis=0)
    last_image = ax.imshow(mean_cm, vmin=0.0, vmax=1.0, cmap='Blues')
    for row in range(2):
        for col in range(2):
            value = mean_cm[row, col]
            ax.text(
                col,
                row,
                f'{value:.2f}',
                ha='center',
                va='center',
                fontsize=13,
                color='white' if value >= 0.55 else 'black',
            )
    ax.set_xticks([0, 1], ['HC', 'PD_OFF'])
    ax.set_yticks([0, 1], ['HC', 'PD_OFF'])
    ax.set_xlabel('Clase predicha')
    ax.set_ylabel('Clase real')
    ax.set_title(result['config']['label'])

fig.colorbar(last_image, ax=axes, fraction=0.025, pad=0.02, label='Proporción')
confusion_path = MODEL_FIG_DIR / 'matrices_confusion.png'
fig.savefig(confusion_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura:', confusion_path)


### 14.3 Curvas de entrenamiento y validación

Diagnóstico de sobreajuste en los tres mejores modelos profundos de A2. Las ejecuciones se
prolongan hasta un horizonte común de 80 épocas solo con fines diagnósticos; la línea vertical
indica la mediana de la época de parada temprana. Estas curvas no modifican las métricas
reportadas.

In [ ]:
# ============================================================================
# HELPERS PARA CURVAS DE ENTRENAMIENTO Y VALIDACION
# ============================================================================
def train_dl_with_history(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = EPOCHS,
    patience: int = PATIENCE,
    lr: float = DEFAULT_HP['lr'],
    weight_decay: float = WEIGHT_DECAY,
    batch_size: int = BATCH_SIZE,
) -> tuple[nn.Module, pd.DataFrame, int, int]:
    """Entrenamiento diagnostico con horizonte fijo y registro de early stopping.

    La optimizacion continua hasta ``epochs`` unicamente para construir curvas
    comparables entre corridas. En paralelo se registra la epoca en la que el
    criterio de parada temprana HABRIA detenido el entrenamiento real.

    Esta funcion se usa solo para la figura de curvas de entrenamiento y no modifica las metricas
    finales obtenidas por ``train_dl``.
    """
    model = model.to(DEVICE)
    X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
    y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
    X_val_tensor = torch.from_numpy(X_val.astype(np.float32)).to(DEVICE)
    y_val_tensor = torch.from_numpy(y_val.astype(np.float32)).to(DEVICE)

    n_pos = float((y_train == 1).sum())
    n_neg = float((y_train == 0).sum())
    positive_weight = torch.tensor([
        max(n_neg / max(n_pos, 1.0), 0.5)
    ]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs,
    )

    drop_last = len(X_train_tensor) > batch_size
    loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        drop_last=drop_last,
    )

    best_state = {
        key: value.detach().clone() for key, value in model.state_dict().items()
    }
    best_val_loss = float('inf')
    best_epoch = 0
    stop_epoch = None
    counter = 0
    history_rows = []

    X_train_eval = torch.from_numpy(X_train.astype(np.float32)).to(DEVICE)
    y_train_eval = torch.from_numpy(y_train.astype(np.float32)).to(DEVICE)

    for epoch_idx in range(epochs):
        model.train()
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            train_logits = model(X_train_eval)
            val_logits = model(X_val_tensor)
            train_loss = float(criterion(train_logits, y_train_eval).item())
            val_loss = float(criterion(val_logits, y_val_tensor).item())
            train_probability = torch.sigmoid(train_logits).cpu().numpy()
            val_probability = torch.sigmoid(val_logits).cpu().numpy()

        current_epoch = epoch_idx + 1
        history_rows.append({
            'epoch': current_epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_auc': _safe_auc(y_train, train_probability),
            'val_auc': _safe_auc(y_val, val_probability),
        })

        # Simula exactamente el criterio de early stopping del pipeline real,
        # pero NO interrumpe esta ejecucion diagnostica.
        if stop_epoch is None:
            if val_loss < best_val_loss - 1e-5:
                best_val_loss = val_loss
                best_epoch = current_epoch
                best_state = {
                    key: value.detach().clone()
                    for key, value in model.state_dict().items()
                }
                counter = 0
            else:
                counter += 1
                if counter >= patience:
                    stop_epoch = current_epoch

    if stop_epoch is None:
        stop_epoch = epochs

    # Se conserva el estado que habria seleccionado el entrenamiento real.
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history_rows), best_epoch, stop_epoch


def collect_training_histories(config: dict) -> pd.DataFrame:
    """Registra historias reproduciendo el preprocesamiento real de cada fold."""
    approach = config['approach']
    repr_key = config['repr']
    X, y, groups, _, _ = get_representation_data(
        MAIN_EXPERIMENT, approach, repr_key,
    )

    histories = []
    for seed in HISTORY_SEEDS:
        set_seed(seed)
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=seed,
        )
        for fold_idx, (idx_train, idx_test) in enumerate(
            splitter.split(X, y, groups)
        ):
            if fold_idx not in HISTORY_FOLDS:
                continue

            if approach == 'A_epoca':
                (
                    X_train_inner,
                    X_val_inner,
                    _,
                    y_train_inner,
                    y_val_inner,
                    _,
                ) = prepare_aepoca_dl_inner_data(
                    X[idx_train],
                    X[idx_test],
                    y[idx_train],
                    groups[idx_train],
                    repr_key,
                    seed,
                    fold_idx,
                )
            else:
                X_outer_train, _, y_outer_train, _, _ = _prepare_fold_data(
                    approach, repr_key, idx_train, idx_test,
                )
                groups_outer_train = groups[idx_train]
                train_inner, val_inner = make_inner_train_val_split(
                    y_outer_train,
                    groups_outer_train,
                    approach,
                    seed,
                    fold_idx,
                )
                X_train_inner = X_outer_train[train_inner]
                X_val_inner = X_outer_train[val_inner]
                y_train_inner = y_outer_train[train_inner]
                y_val_inner = y_outer_train[val_inner]

            model = _build_dl_model(
                config['model'],
                n_channels=X_train_inner.shape[1],
                n_basis=X_train_inner.shape[2],
            )
            _, history, best_epoch, stop_epoch = train_dl_with_history(
                model,
                X_train_inner,
                y_train_inner,
                X_val_inner,
                y_val_inner,
            )
            history['seed'] = seed
            history['fold'] = fold_idx
            history['best_epoch'] = best_epoch
            history['stop_epoch'] = stop_epoch
            history['model'] = config['model']
            history['approach'] = approach
            history['repr'] = repr_key
            histories.append(history)

    return pd.concat(histories, ignore_index=True)


In [ ]:
# ============================================================================
# FIGURA: CURVAS DE ENTRENAMIENTO DE LOS TRES MEJORES MODELOS PROFUNDOS
# ============================================================================
# Tres mejores MODELOS profundos de A_epoca segun la evaluacion.
# Primero se conserva la mejor representacion de cada arquitectura y luego se
# eligen las tres arquitecturas con mayor Balanced Accuracy media por sujeto.
_top_deep_rows = _best_result_row(
    'A_epoca', deep_only=True, per_model=True,
).head(3)
TRAINING_CURVE_CONFIGS = [
    _result_to_config(row)
    for _, row in _top_deep_rows.iterrows()
]

print('Modelos seleccionados para curvas de entrenamiento:')
for cfg in TRAINING_CURVE_CONFIGS:
    print(f"  - {cfg['model']} | {cfg['repr']} | {cfg['title']}")


history_tables = []
for config in TRAINING_CURVE_CONFIGS:
    slug = f"{config['approach']}_{config['repr']}_{config['model']}"
    cache_csv = FIGURE_DATA_DIR / f'history_{slug}_{FIGURE_CACHE_TAG}.csv'
    if cache_csv.exists() and not FORCE_REGENERATE_FIGURES:
        history = pd.read_csv(cache_csv)
    else:
        history = collect_training_histories(config)
        history.to_csv(cache_csv, index=False)
    history['title'] = config['title']
    history_tables.append(history)

history_all = pd.concat(history_tables, ignore_index=True)
history_all.to_csv(
    FIGURE_DATA_DIR / 'curvas_entrenamiento_tres_modelos.csv',
    index=False,
)

fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.4), sharey=True)
for ax, config in zip(axes, TRAINING_CURVE_CONFIGS):
    subset = history_all[history_all['title'] == config['title']].copy()

    summary = subset.groupby('epoch').agg(
        train_mean=('train_auc', 'mean'),
        train_std=('train_auc', 'std'),
        val_mean=('val_auc', 'mean'),
        val_std=('val_auc', 'std'),
        n_runs=('val_auc', 'size'),
    ).reset_index()
    summary[['train_std', 'val_std']] = summary[
        ['train_std', 'val_std']
    ].fillna(0.0)

    # Verificacion:
    # todas las epocas deben resumir exactamente las mismas corridas.
    n_expected = subset[['seed', 'fold']].drop_duplicates().shape[0]
    if not (summary['n_runs'] == n_expected).all():
        raise AssertionError(
            f"{config['title']}: el numero de corridas cambia entre epocas."
        )

    print(
        f"{config['title']}: {n_expected} corridas en todas las epocas "
        f"(1-{int(summary['epoch'].max())})."
    )

    x = summary['epoch'].to_numpy()
    train_mean = summary['train_mean'].to_numpy()
    train_std = summary['train_std'].to_numpy()
    val_mean = summary['val_mean'].to_numpy()
    val_std = summary['val_std'].to_numpy()

    ax.plot(x, train_mean, lw=2.0, label='Entrenamiento')
    ax.fill_between(
        x,
        np.clip(train_mean - train_std, 0.0, 1.0),
        np.clip(train_mean + train_std, 0.0, 1.0),
        alpha=0.12,
    )
    ax.plot(x, val_mean, lw=2.0, label='Validación')
    ax.fill_between(
        x,
        np.clip(val_mean - val_std, 0.0, 1.0),
        np.clip(val_mean + val_std, 0.0, 1.0),
        alpha=0.12,
    )

    stop_table = subset[
        ['seed', 'fold', 'stop_epoch']
    ].drop_duplicates()
    stop_epoch_median = int(round(stop_table['stop_epoch'].median()))

    ax.axvline(
        stop_epoch_median,
        ls='--',
        lw=1.4,
        color='0.35',
        label='Mediana parada temprana',
    )
    ax.set_title(config['title'])
    ax.set_xlabel('Época de entrenamiento')
    ax.set_ylim(0.45, 1.02)
    ax.grid(alpha=0.25, lw=0.5)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('AUC')
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()
training_curve_path = MODEL_FIG_DIR / 'curvas_entrenamiento.png'
fig.savefig(training_curve_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura:', training_curve_path)


### 14.4 Curva de aprendizaje por número de sujetos

Estrategia A2 con representación Cholesky. En cada una de las 5 repeticiones se reserva un
conjunto de prueba fijo de 10 sujetos (5 HC y 5 PD_OFF) y se aumenta el número de sujetos
independientes disponibles para entrenamiento (12, 18, 24, 30, 36 y 38).

In [ ]:
# ============================================================================
# HELPERS PARA CURVAS DE APRENDIZAJE CONTROLADAS (A_epoca, X_chol)
# ============================================================================
def _subject_label_table(
    y_values: np.ndarray,
    group_values: np.ndarray,
) -> pd.DataFrame:
    """Una fila por sujeto con su etiqueta binaria."""
    rows = []
    for subject in np.unique(group_values):
        mask = group_values == subject
        labels = np.unique(y_values[mask])
        if len(labels) != 1:
            raise ValueError(f'Etiquetas inconsistentes para sujeto {subject}.')
        rows.append({'subject': str(subject), 'label': int(labels[0])})
    return pd.DataFrame(rows)


def _fixed_subject_holdout(
    subject_table: pd.DataFrame,
    rng: np.random.Generator,
    n_test_per_class: int = LEARNING_TEST_PER_CLASS,
) -> tuple[np.ndarray, np.ndarray]:
    """Crea train-pool/test estratificados; el test queda fijo dentro de cada repeticion."""
    train_subjects, test_subjects = [], []
    for label in (0, 1):
        candidates = subject_table.loc[
            subject_table['label'] == label, 'subject'
        ].to_numpy(dtype=str)
        test = rng.choice(candidates, size=n_test_per_class, replace=False)
        train = np.setdiff1d(candidates, test)
        test_subjects.extend(test.tolist())
        train_subjects.extend(train.tolist())
    return np.asarray(train_subjects, dtype=str), np.asarray(test_subjects, dtype=str)


def _balanced_subject_sample(
    subject_table: pd.DataFrame,
    candidate_subjects: np.ndarray,
    n_subjects: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Muestra un subconjunto balanceado desde un pool de sujetos ya definido."""
    if n_subjects % 2 != 0:
        raise ValueError('n_subjects debe ser par para muestreo balanceado.')
    candidate_subjects = np.asarray(candidate_subjects, dtype=str)
    n_per_class = n_subjects // 2
    selected = []
    for label in (0, 1):
        class_subjects = subject_table.loc[
            (subject_table['label'] == label)
            & (subject_table['subject'].isin(candidate_subjects)),
            'subject',
        ].to_numpy(dtype=str)
        if len(class_subjects) < n_per_class:
            raise ValueError(
                f'No hay suficientes sujetos clase={label}: '
                f'{len(class_subjects)} < {n_per_class}'
            )
        selected.extend(
            rng.choice(class_subjects, size=n_per_class, replace=False).tolist()
        )
    return np.asarray(selected, dtype=str)


def _sample_epoch_indices_by_subject(
    groups_values: np.ndarray,
    fraction: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """Selecciona aleatoriamente una fraccion de epocas de CADA sujeto.

    Se conserva al menos una epoca por sujeto. Con fraction=1 se conservan todas.
    El muestreo aleatorio se repite en cada repeticion para evitar que la curva
    dependa de una zona temporal concreta del registro.
    """
    if not (0 < fraction <= 1.0):
        raise ValueError('fraction debe pertenecer a (0, 1].')

    selected: list[int] = []
    groups_values = np.asarray(groups_values)
    for subject in np.unique(groups_values):
        idx = np.flatnonzero(groups_values == subject)
        if fraction >= 1.0:
            chosen = idx
        else:
            n_keep = max(1, int(np.ceil(fraction * len(idx))))
            chosen = rng.choice(idx, size=n_keep, replace=False)
        selected.extend(np.asarray(chosen, dtype=int).tolist())
    return np.asarray(sorted(selected), dtype=int)


def _mean_epochs_per_subject(groups_values: np.ndarray) -> float:
    counts = pd.Series(np.asarray(groups_values).astype(str)).value_counts()
    return float(counts.mean())


def _learning_configs_same_representation() -> list[dict]:
    """LogReg + dos mejores DL de A_epoca usando TODOS X_chol.

    Fijar la representacion evita atribuir al modelo una diferencia causada por
    B-spline/Cholesky/FPCA. Los DL se seleccionan por Balanced Accuracy de la
    corrida principal, restringidos a X_chol.
    """
    deep_chol = DF_ALL[
        (DF_ALL['status'] == 'ok')
        & (DF_ALL['nivel'] == 'sujeto')
        & (DF_ALL['scenario'] == 'A_epoca')
        & (DF_ALL['repr'] == 'X_chol')
        & (DF_ALL['model'].isin(DL_MODELS.keys()))
    ].copy()
    deep_chol = deep_chol.sort_values(
        ['balanced_acc_mean', 'auc_mean'], ascending=[False, False]
    ).drop_duplicates(subset=['model'], keep='first')
    if len(deep_chol) < 2:
        raise RuntimeError('Se requieren al menos dos modelos DL con X_chol.')

    top_two = deep_chol.head(2)
    configs = [{
        'label': 'LogReg + Cholesky',
        'model': 'LogReg',
        'approach': 'A_epoca',
        'repr': 'X_chol',
    }]
    for _, row in top_two.iterrows():
        configs.append({
            'label': f"{row['model']} + Cholesky",
            'model': str(row['model']),
            'approach': 'A_epoca',
            'repr': 'X_chol',
        })
    return configs


def _evaluate_learning_holdout(
    config: dict,
    train_subjects: np.ndarray,
    test_subjects: np.ndarray,
    epoch_fraction: float,
    seed: int,
) -> dict:
    """Entrena una configuracion con test fijo y fraccion controlada de epocas.

    - Todos los modelos usan A_epoca + X_chol.
    - El test usa SIEMPRE todas sus epocas y se evalua a nivel sujeto.
    - LogReg ajusta scaler/modelo solo con las epocas seleccionadas de training.
    - DL separa inner-train/inner-val POR SUJETO; solo inner-train se submuestrea.
      El scaler se ajusta exclusivamente con inner-train submuestreado.
    """
    if config['approach'] != 'A_epoca' or config['repr'] != 'X_chol':
        raise ValueError('La curva controlada exige A_epoca + X_chol.')

    X, y, groups, _, _ = get_representation_data(
        MAIN_EXPERIMENT, 'A_epoca', 'X_chol',
    )
    group_strings = groups.astype(str)
    mask_train = np.isin(group_strings, np.asarray(train_subjects, dtype=str))
    mask_test = np.isin(group_strings, np.asarray(test_subjects, dtype=str))
    if np.any(mask_train & mask_test):
        raise AssertionError('Train y test comparten sujetos.')

    X_outer_train, y_outer_train = X[mask_train], y[mask_train]
    g_outer_train = groups[mask_train]
    X_outer_test, y_outer_test = X[mask_test], y[mask_test]
    g_outer_test = groups[mask_test]

    rng = np.random.default_rng(seed)
    model_name = config['model']

    if model_name in ML_MODELS:
        idx_keep = _sample_epoch_indices_by_subject(
            g_outer_train, epoch_fraction, rng,
        )
        X_fit_raw = X_outer_train[idx_keep]
        y_fit = y_outer_train[idx_keep]
        g_fit = g_outer_train[idx_keep]

        X_fit, X_test = standardize_per_fold(X_fit_raw, X_outer_test)
        if model_name == 'LogReg':
            model = LogisticRegression(
                C=1.0,
                class_weight='balanced',
                max_iter=2000,
                random_state=RANDOM_STATE,
            )
        elif model_name == 'RandomForest':
            model = RandomForestClassifier(
                n_estimators=200,
                max_depth=10,
                class_weight='balanced',
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        else:
            raise ValueError(f'Modelo ML no contemplado: {model_name}')

        model.fit(X_fit.reshape(len(X_fit), -1), y_fit)
        probability = model.predict_proba(
            X_test.reshape(len(X_test), -1)
        )[:, 1]
        mean_epochs_train = _mean_epochs_per_subject(g_fit)
        n_train_subjects_effective = len(np.unique(g_fit))

    else:
        idx_inner_train, idx_inner_val = make_inner_train_val_split(
            y_outer_train,
            g_outer_train,
            'A_epoca',
            seed,
            0,
        )
        if set(g_outer_train[idx_inner_train]) & set(g_outer_train[idx_inner_val]):
            raise AssertionError('Inner-train e inner-val comparten sujetos.')

        inner_rng = np.random.default_rng(seed + 100_000)
        local_keep = _sample_epoch_indices_by_subject(
            g_outer_train[idx_inner_train],
            epoch_fraction,
            inner_rng,
        )
        idx_fit = idx_inner_train[local_keep]

        X_fit_raw = X_outer_train[idx_fit]
        y_fit = y_outer_train[idx_fit]
        g_fit = g_outer_train[idx_fit]
        X_val_raw = X_outer_train[idx_inner_val]
        y_val = y_outer_train[idx_inner_val]

        # Fit del scaler SOLO con inner-train submuestreado. Val/test solo transforman.
        X_eval_raw = np.concatenate([X_val_raw, X_outer_test], axis=0)
        X_fit, X_eval = standardize_per_fold(X_fit_raw, X_eval_raw)
        n_val = len(X_val_raw)
        X_val = X_eval[:n_val]
        X_test = X_eval[n_val:]

        model = _build_dl_model(
            model_name,
            n_channels=X_fit.shape[1],
            n_basis=X_fit.shape[2],
        )
        model = train_dl(
            model,
            X_fit,
            y_fit,
            X_val,
            y_val,
            lr=DEFAULT_HP['lr'],
        )
        probability = predict_proba_dl(model, X_test)
        mean_epochs_train = _mean_epochs_per_subject(g_fit)
        n_train_subjects_effective = len(np.unique(g_fit))

    y_subject, p_subject = aggregate_by_subject(
        probability,
        y_outer_test,
        g_outer_test,
    )
    metrics = compute_metrics(y_subject, p_subject)
    return {
        'model': model_name,
        'label': config['label'],
        'repr': 'X_chol',
        'epoch_fraction': float(epoch_fraction),
        'epoch_fraction_pct': int(round(100 * epoch_fraction)),
        'mean_epochs_train_per_subject': mean_epochs_train,
        'n_train_subjects_effective': int(n_train_subjects_effective),
        'n_test_subjects': int(len(np.unique(g_outer_test))),
        **metrics,
    }


In [ ]:
# FIGURA: CURVA DE APRENDIZAJE POR NÚMERO DE SUJETOS
# ============================================================================
# Se mantiene fijo el conjunto de prueba dentro de cada repetición y se aumenta
# únicamente el número de sujetos independientes disponibles para entrenamiento.

LEARNING_CONFIGS = _learning_configs_same_representation()

print('Configuraciones para la curva de aprendizaje:')
for cfg in LEARNING_CONFIGS:
    print(f"  - {cfg['model']} | A_epoca | X_chol")

X_ref, y_ref, g_ref, _, _ = get_representation_data(
    MAIN_EXPERIMENT, 'A_epoca', 'X_chol',
)
subject_table = _subject_label_table(y_ref, g_ref)
if len(subject_table) != 48:
    raise AssertionError(
        f'Se esperaban 48 sujetos y se encontraron {len(subject_table)}.'
    )

holdouts = []
for repeat in range(LEARNING_REPEATS):
    rng_split = np.random.default_rng(30000 + repeat)
    train_pool, test_subjects = _fixed_subject_holdout(
        subject_table,
        rng_split,
        n_test_per_class=LEARNING_TEST_PER_CLASS,
    )
    holdouts.append((train_pool, test_subjects))

subject_raw_path = FIGURE_DATA_DIR / f'curva_aprendizaje_sujetos_registros_{FIGURE_CACHE_TAG}.csv'
subject_summary_path = FIGURE_DATA_DIR / f'curva_aprendizaje_sujetos_resumen_{FIGURE_CACHE_TAG}.csv'

if (
    subject_raw_path.exists()
    and subject_summary_path.exists()
    and not FORCE_REGENERATE_FIGURES
):
    subject_records = pd.read_csv(subject_raw_path)
    subject_summary = pd.read_csv(subject_summary_path)
else:
    rows = []
    for repeat, (train_pool, test_subjects) in enumerate(holdouts):
        for size_index, n_subjects in enumerate(LEARNING_SUBJECT_SIZES):
            rng_subjects = np.random.default_rng(
                50000 + 1000 * repeat + size_index
            )
            selected_train = _balanced_subject_sample(
                subject_table,
                train_pool,
                n_subjects,
                rng_subjects,
            )
            # Misma seleccion de sujetos para todos los modelos.
            data_seed = 60000 + 1000 * repeat + size_index
            for config in LEARNING_CONFIGS:
                set_seed(data_seed)
                row = _evaluate_learning_holdout(
                    config,
                    train_subjects=selected_train,
                    test_subjects=test_subjects,
                    epoch_fraction=1.0,
                    seed=data_seed,
                )
                row['repeat'] = repeat
                row['n_train_subjects_requested'] = n_subjects
                rows.append(row)

    subject_records = pd.DataFrame(rows)
    subject_summary = subject_records.groupby(
        ['n_train_subjects_requested', 'label'], as_index=False,
    ).agg(
        balanced_acc_mean=('balanced_acc', 'mean'),
        balanced_acc_std=('balanced_acc', 'std'),
        auc_mean=('auc', 'mean'),
        auc_std=('auc', 'std'),
    )
    subject_summary[
        ['balanced_acc_std', 'auc_std']
    ] = subject_summary[
        ['balanced_acc_std', 'auc_std']
    ].fillna(0.0)
    subject_records.to_csv(subject_raw_path, index=False)
    subject_summary.to_csv(subject_summary_path, index=False)

print('\nResumen curva por sujetos:')
print(subject_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), sharex=True)
for config in LEARNING_CONFIGS:
    subset = subject_summary[
        subject_summary['label'] == config['label']
    ].sort_values('n_train_subjects_requested')
    x = subset['n_train_subjects_requested'].to_numpy()

    for ax, metric, ylabel in [
        (axes[0], 'balanced_acc', 'Exactitud balanceada'),
        (axes[1], 'auc', 'AUC'),
    ]:
        mean = subset[f'{metric}_mean'].to_numpy()
        std = subset[f'{metric}_std'].to_numpy()
        ax.plot(x, mean, marker='o', lw=2.0, label=config['label'])
        ax.fill_between(
            x,
            np.clip(mean - std, 0.0, 1.0),
            np.clip(mean + std, 0.0, 1.0),
            alpha=0.13,
        )
        ax.set_xlabel('Número de sujetos independientes en entrenamiento')
        ax.set_ylabel(ylabel)
        ax.set_xticks(LEARNING_SUBJECT_SIZES)
        ax.set_ylim(0.45, 1.0)
        ax.grid(alpha=0.25, lw=0.5)
        ax.spines[['top', 'right']].set_visible(False)

axes[0].legend(frameon=False)
fig.suptitle(
    'Efecto de aumentar sujetos independientes\n'
    '(test fijo y misma representacion Cholesky)',
    y=1.03,
)
fig.tight_layout()
subject_curve_path = MODEL_FIG_DIR / 'curva_aprendizaje_sujetos.png'
fig.savefig(subject_curve_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', subject_curve_path)


## 15. Verificación de los productos generados

Se comprueba que existan las figuras y tablas principales del modelado y se guarda un
inventario de archivos (`manifest_modelado.csv`).

In [ ]:
# Verificación de los productos del cuaderno 03.
expected_figures = [
    FIG_DIR / 'componentes_FPCA_B.png',
    FIG_DIR / 'matrices_confusion.png',
    FIG_DIR / 'curvas_entrenamiento.png',
    FIG_DIR / 'curva_aprendizaje_sujetos.png',
]
expected_results = [
    RESULTS_DIR / 'tabla_comparativa_completa.csv',
    RESULTS_DIR / 'tabla_fpca_componentes.csv',
    RESULTS_DIR / 'tabla_fpca_componentes_B.csv',
]

rows = []
for path in expected_figures + expected_results:
    rows.append({
        'archivo': path.name,
        'tipo': 'figura' if path.suffix.lower() == '.png' else 'resultado',
        'ruta': str(path),
        'existe': path.exists(),
        'tamano_bytes': path.stat().st_size if path.exists() else 0,
    })

manifest_modelado = pd.DataFrame(rows)
manifest_modelado.to_csv(OUTPUT_DIR / 'manifest_modelado.csv', index=False)
display(manifest_modelado)

missing = manifest_modelado.loc[~manifest_modelado['existe'], 'archivo'].tolist()
if missing:
    raise RuntimeError(f'Faltan productos del modelado: {missing}')

print('\nCuaderno 03 completado correctamente.')
print('Resultados:', RESULTS_DIR)
print('Figuras:', FIG_DIR)
